# CoachPosture AI — Exploration des données & Modèles

Ce notebook permet d'explorer :
- La distribution des features géométriques
- Les keypoints MediaPipe sur des exemples
- Les performances des modèles entraînés
- L'analyse des seuils de détection d'anomalies

**Pré-requis :** Lancer les scripts de données avant ce notebook.
```bash
python src/data/download_datasets.py --synthetic-only
python src/data/extract_keypoints.py
python src/data/label_postures.py
python src/models/train.py
```

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

%matplotlib inline
plt.rcParams['figure.facecolor'] = '#0e1117'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

ModuleNotFoundError: No module named 'pandas'

## 1. Chargement du dataset labellisé

In [ ]:
labeled_path = ROOT / 'data' / 'labeled' / 'all_labeled.csv'

if labeled_path.exists():
    df = pd.read_csv(labeled_path)
    print(f'Échantillons: {len(df)}')
    print(f'Colonnes: {len(df.columns)}')
    print(f'\nDistribution posture_correcte:')
    print(df['posture_correcte'].value_counts())
    print(f'\nScore moyen: {df["score_posture"].mean():.1f}/100')
    df[['angle_dos', 'angle_tete', 'symetrie_epaules', 'inclinaison_tronc', 'score_posture']].describe()
else:
    print('Dataset non trouvé. Générons des données synthétiques pour la démo...')
    from src.data.label_postures import compute_features
    from src.data.extract_keypoints import LANDMARK_NAMES
    
    # Simulation de 200 postures
    np.random.seed(42)
    rows = []
    for i in range(200):
        row = {}
        cx, cy = 0.50, 0.50
        for name in LANDMARK_NAMES:
            row[f'{name}_x'] = cx + np.random.normal(0, 0.05)
            row[f'{name}_y'] = cy + np.random.normal(0, 0.1)
            row[f'{name}_z'] = 0.0
            row[f'{name}_visibility'] = 0.9
        rows.append(pd.Series(row))
    df_raw = pd.DataFrame(rows)
    features = df_raw.apply(compute_features, axis=1)
    df = pd.concat([df_raw, pd.DataFrame(list(features))], axis=1)
    df['posture_correcte'] = (df['score_posture'] >= 60).astype(int)
    df['extraction_success'] = True
    print(f'Données synthétiques générées: {len(df)} échantillons')

## 2. Distribution des features géométriques

In [ ]:
feature_cols = ['angle_dos', 'angle_tete', 'symetrie_epaules', 'inclinaison_tronc', 'angle_cou']
available = [c for c in feature_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

colors = {'Correcte': '#00e676', 'Anomale': '#f44336'}

for i, feat in enumerate(available):
    ax = axes[i]
    df_ok = df[df['posture_correcte'] == 1][feat].dropna()
    df_bad = df[df['posture_correcte'] == 0][feat].dropna()
    
    if len(df_ok) > 0:
        ax.hist(df_ok, bins=30, alpha=0.7, color=colors['Correcte'], label='Correcte', density=True)
    if len(df_bad) > 0:
        ax.hist(df_bad, bins=30, alpha=0.7, color=colors['Anomale'], label='Anomale', density=True)
    
    ax.set_title(feat.replace('_', ' ').title(), color='white')
    ax.set_xlabel('Valeur (°)', color='white')
    ax.legend()
    ax.grid(True, alpha=0.2)

# Score global
ax = axes[5]
if 'score_posture' in df.columns:
    ax.hist(df[df['posture_correcte']==1]['score_posture'], bins=30, alpha=0.7,
            color=colors['Correcte'], label='Correcte', density=True)
    ax.hist(df[df['posture_correcte']==0]['score_posture'], bins=30, alpha=0.7,
            color=colors['Anomale'], label='Anomale', density=True)
    ax.axvline(x=60, color='yellow', linestyle='--', label='Seuil (60)')
    ax.set_title('Score Posture Global', color='white')
    ax.legend()
    ax.grid(True, alpha=0.2)

plt.suptitle('Distribution des Features Posturales', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé: results/features_distribution.png')

## 3. Visualisation squelette MediaPipe (simulation)

In [ ]:
import cv2
from PIL import Image

# Connexions squelette MediaPipe Pose (paires de joints)
SKELETON_CONNECTIONS = [
    (11, 12),  # Épaules
    (11, 13), (13, 15),  # Bras gauche
    (12, 14), (14, 16),  # Bras droit
    (11, 23), (12, 24),  # Torse
    (23, 24),  # Hanches
    (23, 25), (25, 27),  # Jambe gauche
    (24, 26), (26, 28),  # Jambe droite
    (0, 11), (0, 12),   # Tête vers épaules
]

def draw_stick_figure(ax, landmarks_row, title='Posture', color='green'):
    """Dessine un squelette stylisé depuis une ligne de keypoints."""
    from src.data.extract_keypoints import LANDMARK_NAMES
    
    pts = {}
    for i, name in enumerate(LANDMARK_NAMES):
        x = landmarks_row.get(f'{name}_x', 0.5)
        y = landmarks_row.get(f'{name}_y', 0.5)
        pts[i] = (float(x), 1.0 - float(y))  # Inverser Y pour affichage naturel
    
    for a, b in SKELETON_CONNECTIONS:
        if a in pts and b in pts:
            ax.plot([pts[a][0], pts[b][0]], [pts[a][1], pts[b][1]],
                    color=color, linewidth=2, alpha=0.8)
    
    for idx, (x, y) in pts.items():
        ax.scatter(x, y, c=color, s=30, zorder=5)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(title, color='white', fontsize=12)
    ax.set_aspect('equal')
    ax.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(12, 6))

# Quelques exemples de postures différentes
if len(df) >= 3:
    # Meilleure posture
    best_idx = df['score_posture'].idxmax() if 'score_posture' in df.columns else 0
    worst_idx = df['score_posture'].idxmin() if 'score_posture' in df.columns else len(df)-1
    mid_idx = len(df) // 2
    
    best_score = df.loc[best_idx, 'score_posture'] if 'score_posture' in df.columns else 80
    worst_score = df.loc[worst_idx, 'score_posture'] if 'score_posture' in df.columns else 30
    mid_score = df.loc[mid_idx, 'score_posture'] if 'score_posture' in df.columns else 60
    
    draw_stick_figure(axes[0], df.loc[best_idx], 
                      f'Meilleure ({best_score:.0f}/100)', color='#00e676')
    draw_stick_figure(axes[1], df.loc[mid_idx], 
                      f'Moyenne ({mid_score:.0f}/100)', color='#ffeb3b')
    draw_stick_figure(axes[2], df.loc[worst_idx], 
                      f'Pire ({worst_score:.0f}/100)', color='#f44336')

plt.suptitle('Exemples de Postures — Squelette MediaPipe', fontsize=14)
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'skeleton_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé: results/skeleton_examples.png')

## 4. Analyse de l'Autoencoder — Espace latent

In [ ]:
import pickle
from src.models.train import PostureAutoencoder

MODELS_DIR = ROOT / 'data' / 'processed' / 'models'
ae_path = MODELS_DIR / 'autoencoder.pt'
scaler_path = MODELS_DIR / 'scaler.pkl'

FEATURE_COLS = ['angle_dos', 'angle_tete', 'symetrie_epaules',
                'inclinaison_tronc', 'angle_cou', 'ratio_epaules_hanches']

if ae_path.exists() and scaler_path.exists():
    checkpoint = torch.load(ae_path, map_location='cpu')
    ae = PostureAutoencoder(checkpoint['input_dim'])
    ae.load_state_dict(checkpoint['model_state_dict'])
    ae.eval()
    threshold = checkpoint['threshold']
    print(f'Autoencoder chargé — threshold: {threshold:.6f}')
    
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)
    
    # Features disponibles
    available_feats = [c for c in FEATURE_COLS if c in df.columns]
    X = df[available_feats].dropna().values.astype(np.float32)
    X_sc = scaler.transform(X[:, :scaler.n_features_in_]) if hasattr(scaler, 'n_features_in_') else X
    
    X_t = torch.tensor(X_sc, dtype=torch.float32)
    with torch.no_grad():
        errors = ae.reconstruction_error(X_t).numpy()
        latent = ae.encoder(X_t).numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribution des erreurs de reconstruction
    ax = axes[0]
    y_labels = df.loc[df[available_feats].notna().all(axis=1), 'posture_correcte'].values[:len(errors)]
    
    if len(y_labels) == len(errors):
        ax.hist(errors[y_labels==1], bins=50, alpha=0.7, color='#00e676', 
                label='Posture correcte', density=True)
        ax.hist(errors[y_labels==0], bins=50, alpha=0.7, color='#f44336', 
                label='Anomalie', density=True)
    else:
        ax.hist(errors, bins=50, alpha=0.7, color='#00e5ff', density=True)
    
    ax.axvline(x=threshold, color='yellow', linestyle='--', linewidth=2,
               label=f'Seuil ({threshold:.4f})')
    ax.set_title('Erreurs de reconstruction (Autoencoder)', color='white')
    ax.set_xlabel('Erreur MSE')
    ax.legend()
    ax.grid(True, alpha=0.2)
    
    # PCA de l'espace latent (2D)
    from sklearn.decomposition import PCA
    ax = axes[1]
    if latent.shape[0] > 2:
        pca = PCA(n_components=2)
        latent_2d = pca.fit_transform(latent)
        if len(y_labels) == len(latent_2d):
            scatter = ax.scatter(latent_2d[:, 0], latent_2d[:, 1],
                                c=y_labels, cmap='RdYlGn', alpha=0.6, s=20)
            plt.colorbar(scatter, ax=ax, label='0=Anomale, 1=Correct')
        else:
            ax.scatter(latent_2d[:, 0], latent_2d[:, 1], color='#00e5ff', alpha=0.5, s=15)
        var_explained = pca.explained_variance_ratio_.sum() * 100
        ax.set_title(f'Espace latent — PCA ({var_explained:.1f}% variance)', color='white')
        ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'autoencoder_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Autoencoder non entraîné. Lancez : python src/models/train.py')

## 5. Comparaison des 3 modèles

In [ ]:
results_json = ROOT / 'results' / 'evaluation_results.json'

if results_json.exists():
    import json
    with open(results_json) as f:
        all_evals = json.load(f)
    
    last_eval = all_evals[-1]['evaluations']
    
    models = [e['model'] for e in last_eval]
    aucs = [e.get('roc_auc', 0) or 0 for e in last_eval]
    f1s = [e['classification_report'].get('weighted avg', {}).get('f1-score', 0) for e in last_eval]
    accs = [e['classification_report'].get('accuracy', 0) for e in last_eval]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(models))
    width = 0.25
    
    bars1 = ax.bar(x - width, accs, width, label='Accuracy', color='#00e5ff', alpha=0.8)
    bars2 = ax.bar(x, f1s, width, label='F1-Score', color='#00e676', alpha=0.8)
    bars3 = ax.bar(x + width, aucs, width, label='ROC-AUC', color='#ff9800', alpha=0.8)
    
    def autolabel(bars):
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', 
                        va='bottom', fontsize=9, color='white')
    autolabel(bars1); autolabel(bars2); autolabel(bars3)
    
    ax.set_ylim(0, 1.1)
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_title('Comparaison des Modèles — CoachPosture AI', color='white', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.2, axis='y')
    
    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Sauvegardé: results/model_comparison.png')
else:
    print('Résultats d\'évaluation non trouvés. Lancez : python src/models/evaluate.py')

## 6. Test de l'agent Ollama

In [ ]:
from src.agent.posture_agent import PostureAgent, PostureFeatures

agent = PostureAgent(model='qwen2.5:7b')

# Exemple de mauvaise posture
features = PostureFeatures(
    angle_dos=35.0,        # Dos courbé
    angle_tete=28.0,       # Tête penchée en avant
    symetrie_epaules=7.5,  # Épaules asymétriques
    inclinaison_tronc=15.0,
    angle_cou=32.0,
    score_posture=38.0,
)

print('Envoi à Ollama...')
rec = agent.recommend(features, alert_duration=25.0)

print(f'\n=== Recommandation ({rec.model_utilise}) ===')
print(f'Problème: {rec.probleme_principal}')
print(f'Conseil: {rec.recommandation}')
print(f'Exercice: {rec.exercice_suggere}')
print(f'Score: {rec.score_posture:.0f}/100')

agent.close()